# C6 example 4/4: `RestrictedWETensorProduct` with internal spherical harmonics

This notebook keeps the same physical features and hidden architecture, but uses e3nn's 3D spherical harmonics restricted to the planar C6 subgroup of O(3). C6 acts by rotations around z. Thus each input/output 3D vector is still explicitly represented as $(x,y)\in E_1$ plus $z\in A$.

The important API difference is that `RestrictedWETensorProduct.forward_from_points` accepts the geometric 3D point directly. It owns a `RestrictedSphericalHarmonics` evaluator and computes the filter internally:

$$r\xrightarrow{Y_0\oplus\cdots\oplus Y_3}Y(r)\xrightarrow{\mathrm{RestrictedWETP}}\text{features}.$$

The default C6 full bandlimit is $L_{full}=3$, so degrees $l=0,1,2,3$ are included.

In [ ]:
import torch
from we3nn import CyclicGroup, RestrictedSphericalHarmonics, nn

torch.manual_seed(7)
torch.set_printoptions(precision=5, sci_mode=False)
G = CyclicGroup(6)
A = G.trivial_representation
E1 = G.standard_representation
regular = G.regular_representation()
input_rep = 3 * E1 + 5 * A
hidden_rep = 2 * regular
output_rep = E1 + 4 * A
spherical = RestrictedSphericalHarmonics(
    G, degrees=None, normalization='component', basis='o3'
)
print('spherical degrees:', spherical.degrees)
print('restricted filter representation:', spherical.rep_out.name)
print('dimensions:', input_rep.size, 'x', spherical.rep_out.size, '->', hidden_rep.size, '->', output_rep.size)

In [ ]:
def pack_input(vectors, scalars):
    xy = vectors[..., :, :2].reshape(*vectors.shape[:-2], 6)
    return torch.cat((xy, vectors[..., :, 2], scalars), dim=-1)

def unpack_input(x):
    xy = x[..., :6].reshape(*x.shape[:-1], 3, 2)
    return torch.cat((xy, x[..., 6:9].unsqueeze(-1)), dim=-1), x[..., 9:11]

def unpack_output(y):
    return torch.cat((y[..., :2], y[..., 2:3]), dim=-1), y[..., 3:6]

def rotate_points(points, element):
    Rxy = E1(element).to(device=points.device, dtype=points.dtype)
    return torch.cat((points[..., :2] @ Rxy.T, points[..., 2:3]), dim=-1)

vectors = torch.tensor([[[1.0, 0.2, -0.4], [-0.3, 0.8, 1.2], [0.5, -0.7, 0.1]]])
scalars = torch.tensor([[0.6, -1.1]])
point = torch.tensor([[0.8, 0.35, -0.2]])
x = nn.RepresentationTensor(pack_input(vectors, scalars), input_rep)
Y = spherical(point)
print('point:', point)
print('internally used restricted spherical harmonics Y_l(r):', Y)
print('harmonic block sizes:', [2*l + 1 for l in spherical.degrees])

## Architecture

The contraction has the same Wigner--Eckart form as notebook 3,

$$z_o=\sum_p w_p(\lVert r\rVert,r_z)(C_p)_{oij}x_iY_j(r),$$

but `forward_from_points` evaluates e3nn spherical harmonics inside the layer. The $C_p$ basis spans the complete finite-C6 Hom space after restriction; it is not limited to parent-O(3) coupling paths. The radial networks use only C6-invariant quantities.

In [ ]:
class RestrictedHarmonicNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.harmonics = spherical
        self.input_layer = nn.RestrictedWETensorProduct(
            input_rep, self.harmonics, hidden_rep, shared_weights=False
        )
        self.activation = nn.PointActiv(hidden_rep, torch.relu)
        self.output_layer = nn.RestrictedWETensorProduct(
            hidden_rep, self.harmonics, output_rep, shared_weights=False
        )
        self.radial_in = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.input_layer.weight_numel),
        )
        self.radial_out = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.output_layer.weight_numel),
        )

    def forward(self, features, points):
        invariant_geometry = torch.stack(
            (torch.linalg.vector_norm(points, dim=-1), points[..., 2]), dim=-1
        )
        w_in = self.radial_in(invariant_geometry)
        w_out = self.radial_out(invariant_geometry)
        # Spherical filter evaluation happens inside both calls.
        h_pre = self.input_layer.forward_from_points(features, points, w_in)
        h = self.activation(h_pre)
        y = self.output_layer.forward_from_points(h, points, w_out)
        return y, w_in, w_out, h_pre, h

model = RestrictedHarmonicNetwork().eval()
y, w_in, w_out, h_pre, h = model(x, point)
print('input/output reduced-weight counts:', model.input_layer.weight_numel, model.output_layer.weight_numel)
print('hidden before PointActiv:', h_pre.tensor)
print('hidden after  PointActiv:', h.tensor)
print('physical output (vector, scalars):', unpack_output(y.tensor))
kernel_basis = model.input_layer.sample_kernel_basis(point)
print('sampled input-kernel basis shape [batch, paths, out, in]:', tuple(kernel_basis.shape))

## All six simultaneous rotations

The input feature fiber and the 3D point are rotated together. We print the spherical harmonics at every rotated point, all physical inputs and outputs, and the direct equivariance residual.

In [ ]:
errors = []
for k, element in enumerate(G.elements):
    x_k = x.transform_fibers(element)
    point_k = rotate_points(point, element)
    y_k, w_in_k, w_out_k, _, _ = model(x_k, point_k)
    expected_k = y.transform_fibers(element)
    Y_k = spherical(point_k)
    error = (y_k.tensor - expected_k.tensor).abs().max().item()
    errors.append(error)
    torch.testing.assert_close(y_k.tensor, expected_k.tensor, atol=8e-5, rtol=8e-5)
    torch.testing.assert_close(w_in_k, w_in, atol=1e-6, rtol=1e-6)
    torch.testing.assert_close(w_out_k, w_out, atol=1e-6, rtol=1e-6)
    in_vectors_k, in_scalars_k = unpack_input(x_k.tensor)
    out_vector_k, out_scalars_k = unpack_output(y_k.tensor)
    print(f'rotation {k}: angle={60*k:3d} degrees')
    print('  rotated geometry:', point_k[0].tolist())
    print('  spherical Y     :', Y_k[0].tolist())
    print('  input vectors   :', in_vectors_k[0].tolist())
    print('  input scalars   :', in_scalars_k[0].tolist())
    print('  output vector   :', out_vector_k[0].tolist())
    print('  output scalars  :', out_scalars_k[0].tolist())
    print(f'  max equivariance error: {error:.3e}')

print('maximum over all rotations:', max(errors))

## Visualizing restricted spherical harmonics and the sampled kernel

The spherical-harmonic heatmap groups all 16 real components by degree: $1+3+5+7$ components for $l=0,1,2,3$. The effective-kernel panel explicitly contracts the sampled finite-group kernel paths with the radial coefficients, $K(r)=\sum_p w_p(r)K_p(r)$, and verifies that the first layer produces $K(r)x$.

In [ ]:
import matplotlib.pyplot as plt

hidden_by_rotation, harmonic_by_rotation = [], []
output_vectors, output_scalars = [], []
for element in G.elements:
    x_k = x.transform_fibers(element)
    point_k = rotate_points(point, element)
    y_k, _, _, _, h_k = model(x_k, point_k)
    vector_k, scalars_k = unpack_output(y_k.tensor)
    hidden_by_rotation.append(h_k.tensor[0].detach())
    harmonic_by_rotation.append(spherical(point_k)[0].detach())
    output_vectors.append(vector_k[0].detach())
    output_scalars.append(scalars_k[0].detach())
hidden_by_rotation = torch.stack(hidden_by_rotation).cpu()
harmonic_by_rotation = torch.stack(harmonic_by_rotation).cpu()
output_vectors = torch.stack(output_vectors).cpu()
output_scalars = torch.stack(output_scalars).cpu()
angles_deg = torch.arange(6) * 60
colors = plt.cm.hsv(torch.linspace(0, 5/6, 6).numpy())
component_labels = [f'l={degree}[{component}]' for degree in spherical.degrees for component in range(2 * degree + 1)]
degree_boundaries = torch.tensor([sum(2 * degree + 1 for degree in spherical.degrees[:end]) for end in range(1, len(spherical.degrees))])

basis_at_point = model.input_layer.sample_kernel_basis(point)[0].detach()
effective_kernel = torch.einsum('p,poi->oi', w_in[0].detach(), basis_at_point).cpu()
torch.testing.assert_close(h_pre.tensor[0], effective_kernel @ x.tensor[0], atol=3e-5, rtol=3e-5)

fig = plt.figure(figsize=(19, 10), constrained_layout=True)
ax_in = fig.add_subplot(2, 3, 1, projection='3d')
for index, vector in enumerate(vectors[0]):
    ax_in.quiver(0, 0, 0, *vector.tolist(), color=f'C{index}', linewidth=2, label=f'input v{index+1}')
ax_in.quiver(0, 0, 0, *point[0].tolist(), color='black', linestyle='--', linewidth=2, label='geometry r')
ax_in.set(xlabel='x', ylabel='y', zlabel='z', title='Initial vectors and 3D geometry')
setup_limit = 1.15 * torch.cat((vectors[0], point)).abs().max().item()
ax_in.set_xlim(-setup_limit, setup_limit); ax_in.set_ylim(-setup_limit, setup_limit); ax_in.set_zlim(-setup_limit, setup_limit); ax_in.set_box_aspect((1, 1, 1))
ax_in.legend(fontsize=8)

ax_harm = fig.add_subplot(2, 3, 2)
harmonic_image = ax_harm.imshow(harmonic_by_rotation, aspect='auto', cmap='coolwarm')
for boundary in degree_boundaries.tolist():
    ax_harm.axvline(boundary - 0.5, color='white', linewidth=2)
ax_harm.set(xticks=range(len(component_labels)), xticklabels=component_labels, yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='real spherical-harmonic component', ylabel='C6 rotation', title='Restricted spherical harmonics l=0,1,2,3')
ax_harm.tick_params(axis='x', labelrotation=90, labelsize=7)
fig.colorbar(harmonic_image, ax=ax_harm, shrink=0.75)

ax_hidden = fig.add_subplot(2, 3, 3)
hidden_image = ax_hidden.imshow(hidden_by_rotation, aspect='auto', cmap='coolwarm')
ax_hidden.axvline(5.5, color='white', linewidth=2)
ax_hidden.set(xticks=range(12), yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='regular coordinate (copies 1 | 2)', ylabel='rotation', title='Hidden 2 Reg(C6) after PointActiv')
fig.colorbar(hidden_image, ax=ax_hidden, shrink=0.75)

ax_kernel = fig.add_subplot(2, 3, 4)
kernel_image = ax_kernel.imshow(effective_kernel, aspect='auto', cmap='coolwarm')
ax_kernel.set(xlabel='input coordinate', ylabel='hidden coordinate', title='Restricted-WE effective kernel K(r)')
fig.colorbar(kernel_image, ax=ax_kernel, shrink=0.75)

ax_vec = fig.add_subplot(2, 3, 5, projection='3d')
for k, (vector, color) in enumerate(zip(output_vectors, colors)):
    ax_vec.quiver(0, 0, 0, *vector.tolist(), color=color, linewidth=2, label=f'{60*k}°')
ax_vec.set(xlabel='x', ylabel='y', zlabel='z', title='RestrictedWETensorProduct output vector')
output_limit = max(1e-3, 1.15 * output_vectors.abs().max().item())
ax_vec.set_xlim(-output_limit, output_limit); ax_vec.set_ylim(-output_limit, output_limit); ax_vec.set_zlim(-output_limit, output_limit); ax_vec.set_box_aspect((1, 1, 1))
ax_vec.legend(ncols=2, fontsize=7)

ax_scalar = fig.add_subplot(2, 3, 6)
for channel in range(3):
    ax_scalar.plot(angles_deg, output_scalars[:, channel], marker='o', label=f'output scalar {channel+1}')
ax_scalar.set(xticks=angles_deg.tolist(), xlabel='C6 rotation', ylabel='value', title='Restricted-WE output scalars')
ax_scalar.grid(alpha=0.3); ax_scalar.legend()
plt.show()